## Build the Experiment

- uses the 01-Translation folder experiment files to build the html files for each experiment
- uses the 02-Stimuli folder to grab the stimuli lists for each task 
- creates the separate .jas files for each experiment
- creates the .jzip for importing 



## Libraries and Functions

In [94]:
import os
import re
import json
import shutil
from pathlib import Path
import pandas as pd
import json
import re

def translate_logic_strings(html, translation_dict):

    pattern = r'(===\s*)(["\'])(.*?)(\2)'

    def replace_match(match):
        prefix = match.group(1)
        quote = match.group(2)
        inner = match.group(3)
        end_quote = match.group(4)

        # Only translate if it's in dictionary
        if inner in translation_dict:
            tgt = translation_dict[inner]

            # 🔥 REMOVE inner quotes here
            tgt = tgt.replace('"', '')
            return f'{prefix}{quote}{tgt}{end_quote}'

        return match.group(0)

    html = re.sub(pattern, replace_match, html)

    return html

def fix_next_only(html):

    # -------------------------
    # 1) Fix JATOS function
    # jatos.start<something>Component → jatos.startNextComponent
    # -------------------------
    html = re.sub(
        r'jatos\.start[^C]*Component',
        'jatos.startNextComponent',
        html
    )

    # -------------------------
    # 2) Fix corrupted key names
    # page<something>Text → pageNextText (ONLY if it was supposed to be Next)
    # -------------------------
    html = re.sub(
        r'page[^P]*Text',
        'pageNextText',
        html
    )

    return html

def read_text(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read()


def write_text(path, text):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)


def load_translation_dict(csv_path, english_col="English", translation_col="translation"):
    df = pd.read_csv(csv_path)

    if english_col not in df.columns:
        english_col = df.columns[0]

    if translation_col not in df.columns:
        # fallback: second column
        non_eng = [c for c in df.columns if c != english_col]
        if not non_eng:
            raise ValueError("Could not find translation column in translation CSV")
        translation_col = non_eng[0]

    df = df[[english_col, translation_col]].dropna()
    df[english_col] = df[english_col].astype(str).str.strip()
    df[translation_col] = df[translation_col].astype(str).str.strip()

    return dict(zip(df[english_col], df[translation_col]))

def replace_in_text(text, translation_dict):
    items = sorted(translation_dict.items(), key=lambda x: -len(x[0]))
    for src, tgt in items:
        if not src or not tgt:
            continue
        tgt = tgt.replace('"', '')
        text = text.replace(src, tgt)
    return text


def apply_translations(html, translation_dict):

    # -------------------------
    # 1) Translate html: `...` (SurveyJS HTML blocks)
    # -------------------------
    template_pattern = r'(html\s*:\s*)`([\s\S]*?)`'

    def replace_template(match):
        prefix = match.group(1)
        inner = match.group(2)
        new_inner = replace_in_text(inner, translation_dict)
        return f"{prefix}`{new_inner}`"

    html = re.sub(template_pattern, replace_template, html)

    # -------------------------
    # 2) Translate choices arrays
    # -------------------------
    choices_pattern = r'(\bchoices\s*:\s*\[)([\s\S]*?)(\])'

    def replace_choices(match):
        start = match.group(1)
        body = match.group(2)
        end = match.group(3)

        string_pattern = r'(["\'])(.*?)(\1)'

        def replace_choice_string(m):
            quote = m.group(1)
            inner = m.group(2)
            end_quote = m.group(3)
            new_inner = replace_in_text(inner, translation_dict)
            return f"{quote}{new_inner}{end_quote}"

        new_body = re.sub(string_pattern, replace_choice_string, body)
        return f"{start}{new_body}{end}"

    html = re.sub(choices_pattern, replace_choices, html)

    # -------------------------
    # 3) Translate title fields ONLY
    # -------------------------
    title_pattern = r'(\btitle\s*:\s*)(["\'])(.*?)(\2)'

    def replace_title(match):
        prefix = match.group(1)
        quote = match.group(2)
        inner = match.group(3)
        end_quote = match.group(4)
        new_inner = replace_in_text(inner, translation_dict)
        return f"{prefix}{quote}{new_inner}{end_quote}"

    html = re.sub(title_pattern, replace_title, html)

    # -------------------------
    # 3) Translate simple property values
    # -------------------------
    prop_pattern = r'(\b[A-Za-z_][A-Za-z0-9_]*\b\s*:\s*)(["\'])(.*?)(\2)'

    def replace_property(match):
        prefix = match.group(1)
        quote = match.group(2)
        inner = match.group(3)
        end_quote = match.group(4)

        # only translate if it's in dictionary
        if inner in translation_dict:
            new_inner = translation_dict[inner]
            return f"{prefix}{quote}{new_inner}{end_quote}"

        return match.group(0)

    html = re.sub(prop_pattern, replace_property, html)

    # -------------------------
    # 4) Translate HTML text between tags
    # -------------------------
    html_text_pattern = r'>([^<>]+)<'

    def replace_html_text(match):
        inner = match.group(1)

        if not inner.strip():
            return match.group(0)

        new_inner = replace_in_text(inner, translation_dict)
        return f">{new_inner}<"

    # Apply repeatedly until no more changes (handles nesting)
    prev_html = None
    while prev_html != html:
        prev_html = html
        html = re.sub(html_text_pattern, replace_html_text, html)

    # -------------------------
    # 4) Translate inline UI messages (e.g., error messages)
    # -------------------------
    ui_string_pattern = r'(["\'])([^"\']{5,})(\1)'

    def replace_ui_string(match):
        quote = match.group(1)
        inner = match.group(2)
        end_quote = match.group(3)

        # heuristic: translate only sentences (contains space)
        if " " not in inner:
            return match.group(0)

        new_inner = replace_in_text(inner, translation_dict)

        # if nothing changed, keep original
        if new_inner == inner:
            return match.group(0)

        return f"{quote}{new_inner}{end_quote}"

    html = re.sub(ui_string_pattern, replace_ui_string, html)

    return html


def insert_words_into_html_string(html, word_list, var_name="sampleWords"):
    words_js = json.dumps(word_list, ensure_ascii=False, indent=2)
    replacement = f"const {var_name} = {words_js};"

    pattern = rf"const\s+{re.escape(var_name)}\s*=\s*\[.*?\];"
    new_html, n = re.subn(pattern, replacement, html, flags=re.DOTALL)

    if n == 0:
        raise ValueError(f"Could not find `const {var_name} = [...]` block.")

    return new_html


def load_word_list(csv_path):
    df = pd.read_csv(csv_path)
    if "word" not in df.columns:
        raise ValueError(f"{csv_path} must contain a 'word' column")

    return (
        df["word"]
        .dropna()
        .astype(str)
        .str.strip()
        .replace("", pd.NA)
        .dropna()
        .tolist()
    )


def copy_static_files(src_dir, dst_dir, exclude=("index.html", "consent.html")):
    src_dir = Path(src_dir)
    dst_dir = Path(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)

    for item in src_dir.iterdir():
        if item.name in exclude:
            continue

        target = dst_dir / item.name
        if item.is_dir():
            if target.exists():
                shutil.rmtree(target)
            shutil.copytree(item, target)
        else:
            shutil.copy2(item, target)


def build_task_htmls_from_translation_csv(
    lang,
    translation_csv,
    task_template_root="03-Tasks",
    stimuli_root="02-Stimuli",
    output_root="03-Tasks",
    tasks=("aoa", "image", "concrete", "valence", "arousal", "familiar"),
):
    translation_dict = load_translation_dict(translation_csv)

    summary = []

    for task in tasks:
        task_dir = Path(task_template_root) / task
        index_path = task_dir / "index.html"
        consent_path = task_dir / "consent.html"

        if not index_path.exists():
            raise FileNotFoundError(f"Missing task template: {index_path}")
        if not consent_path.exists():
            raise FileNotFoundError(f"Missing consent template: {consent_path}")

        base_index_html = read_text(index_path)
        base_consent_html = read_text(consent_path)

        # translate both
        translated_index_html = apply_translations(base_index_html, translation_dict)
        translated_index_html = fix_next_only(translated_index_html)
        translated_index_html = translate_logic_strings(translated_index_html, translation_dict)

        translated_consent_html = apply_translations(base_consent_html, translation_dict)
        translated_consent_html = fix_next_only(translated_consent_html)

        stimuli_dir = Path(stimuli_root) / lang / task
        if not stimuli_dir.exists():
            raise FileNotFoundError(f"Missing stimuli folder: {stimuli_dir}")

        stimuli_files = sorted(stimuli_dir.glob("*.csv"))
        if not stimuli_files:
            print(f"No stimuli files found for {task} in {stimuli_dir}")
            continue

        for stimuli_file in stimuli_files:
            words = load_word_list(stimuli_file)

            exp_name = stimuli_file.stem
            exp_dir = Path(output_root) / lang / task / exp_name
            exp_dir.mkdir(parents=True, exist_ok=True)

            # copy assets first
            copy_static_files(task_dir, exp_dir)

            # insert task words
            filled_index_html = insert_words_into_html_string(translated_index_html, words)

            # write translated files
            write_text(exp_dir / "index.html", filled_index_html)
            write_text(exp_dir / "consent.html", translated_consent_html)

            summary.append(
                {
                    "task": task,
                    "stimuli_file": str(stimuli_file),
                    "output_dir": str(exp_dir),
                    "n_words": len(words),
                }
            )

    return pd.DataFrame(summary)

### Build Ukr

In [95]:
summary_df = build_task_htmls_from_translation_csv(
    lang="ukr",
    translation_csv="../01-Translation/ukr/ukr_experiment.csv",
    task_template_root="../03-Tasks",
    stimuli_root="../02-Stimuli",
    output_root="../03-Tasks/builds"
)

summary_df

,task,stimuli_file,output_dir,n_words
0,aoa,../02-Stimuli/ukr/aoa/aoa_list_1.csv,../03-Tasks/builds/ukr/aoa/aoa_list_1,400
1,aoa,../02-Stimuli/ukr/aoa/aoa_list_2.csv,../03-Tasks/builds/ukr/aoa/aoa_list_2,400
2,aoa,../02-Stimuli/ukr/aoa/aoa_list_3.csv,../03-Tasks/builds/ukr/aoa/aoa_list_3,400
3,aoa,../02-Stimuli/ukr/aoa/aoa_list_4.csv,../03-Tasks/builds/ukr/aoa/aoa_list_4,400
4,aoa,../02-Stimuli/ukr/aoa/aoa_list_5.csv,../03-Tasks/builds/ukr/aoa/aoa_list_5,339
5,image,../02-Stimuli/ukr/image/image_list_1.csv,../03-Tasks/builds/ukr/image/image_list_1,400
6,image,../02-Stimuli/ukr/image/image_list_2.csv,../03-Tasks/builds/ukr/image/image_list_2,400
7,image,../02-Stimuli/ukr/image/image_list_3.csv,../03-Tasks/builds/ukr/image/image_list_3,400
8,image,../02-Stimuli/ukr/image/image_list_4.csv,../03-Tasks/builds/ukr/image/image_list_4,400
9,image,../02-Stimuli/ukr/image/image_list_5.csv,../03-Tasks/builds/ukr/image/image_list_5,339


## Update JAS File

In [97]:
import json
import uuid
import shutil
from pathlib import Path

def update_jas_file(jas_path, new_title):
    with open(jas_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # -------------------------
    # 1) Update study-level fields
    # -------------------------
    if "title" in data:
        data["title"] = new_title

    if "uuid" in data:
        data["uuid"] = str(uuid.uuid4())

    # -------------------------
    # 2) Update components (CRITICAL)
    # -------------------------
    if "components" in data:
        for comp in data["components"]:
            if "uuid" in comp:
                comp["uuid"] = str(uuid.uuid4())

            if "title" in comp:
                comp["title"] = f"{new_title}_component"

    # -------------------------
    # 3) Some files nest study inside "study"
    # -------------------------
    if "study" in data:
        study = data["study"]

        if "title" in study:
            study["title"] = new_title

        if "uuid" in study:
            study["uuid"] = str(uuid.uuid4())

        if "components" in study:
            for comp in study["components"]:
                if "uuid" in comp:
                    comp["uuid"] = str(uuid.uuid4())

                if "title" in comp:
                    comp["title"] = f"{new_title}_component"

    # -------------------------
    # Save back
    # -------------------------
    with open(jas_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

def update_all_jas_files(
    base_dir,
    lang=None,
):
    base_path = Path(base_dir)

    jas_files = list(base_path.rglob("*.jas"))

    print(f"Found {len(jas_files)} .jas files")

    for jas_path in jas_files:

        # Build a unique title from folder structure
        parts = jas_path.parts

        # Example: ... / uk / valence / valence_list_1 / study.jas
        try:
            lang_part = parts[-4]
            task_part = parts[-3]
            exp_part = parts[-2]

            new_title = f"{lang_part}_{task_part}_{exp_part}"

        except:
            # fallback if structure is different
            new_title = jas_path.stem

        update_jas_file(jas_path, new_title)

        print(f"Updated: {jas_path} → {new_title}")

In [98]:
update_all_jas_files("../03-Tasks/builds")



Found 30 .jas files
Updated: ../03-Tasks/builds/ukr/concrete/concrete_list_1/concrete10137725508687069911.jas → ukr_concrete_concrete_list_1
Updated: ../03-Tasks/builds/ukr/concrete/concrete_list_5/concrete10137725508687069911.jas → ukr_concrete_concrete_list_5
Updated: ../03-Tasks/builds/ukr/concrete/concrete_list_2/concrete10137725508687069911.jas → ukr_concrete_concrete_list_2
Updated: ../03-Tasks/builds/ukr/concrete/concrete_list_3/concrete10137725508687069911.jas → ukr_concrete_concrete_list_3
Updated: ../03-Tasks/builds/ukr/concrete/concrete_list_4/concrete10137725508687069911.jas → ukr_concrete_concrete_list_4
Updated: ../03-Tasks/builds/ukr/image/image_list_2/image16985989996133131515.jas → ukr_image_image_list_2
Updated: ../03-Tasks/builds/ukr/image/image_list_5/image16985989996133131515.jas → ukr_image_image_list_5
Updated: ../03-Tasks/builds/ukr/image/image_list_4/image16985989996133131515.jas → ukr_image_image_list_4
Updated: ../03-Tasks/builds/ukr/image/image_list_3/image1